# Laboratorio 8 — Desbalance: dónde se remuestrea

Las técnicas de corrección del desbalance ya están escritas y a la vista: ponderar la
pérdida, descartar mayoritarios, generar sintéticos con SMOTE, mover el umbral. Este
laboratorio no las reimplementa. Se ocupa de la pregunta que queda cuando ya se sabe
aplicarlas: **en qué punto del procedimiento corresponde hacerlo**, y qué se paga por
equivocarse.

La respuesta corta es que el remuestreo pertenece **al interior** de la partición: después de
separar los datos de evaluación, nunca antes.

Lo que debemos hacer:

1. Evaluar SMOTE **remuestreando antes de partir**, que es el orden que aparece en mucho
   código publicado.
2. Evaluar la misma estrategia **partiendo antes de remuestrear**, y medir la diferencia.
3. Comparar cuatro estrategias **dentro de la validación cruzada**, que es el protocolo
   correcto, y decidir con la matriz de costos.
4. **Reparar la escala** del modelo remuestreado, y comprobar que ninguna reparación toca
   el ordenamiento.

Hay **cuatro ejercicios** y una recomendación final. Las celdas que
contienen `raise NotImplementedError()` deben completarse; el resto es andamiaje y se ejecuta
sin modificar.

## 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             roc_auc_score)

SEMILLA = 42
COSTO_FN = 500.0   # perdida media por fraude no detectado, USD
COSTO_FP = 5.0     # costo de contactar al titular por una alerta falsa, USD

np.set_printoptions(suppress=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

AZUL, TURQUESA, NARANJA = "#19326E", "#50ACB0", "#CD742A"
VIOLETA, OLIVA, GRIS = "#A3477F", "#89A943", "#8A8A8A"

### El caso

**Credit Card Fraud Detection**, de la Université Libre de Bruxelles: 284 807 transacciones
con tarjeta de crédito de titulares europeos, de las cuales **492 son fraudulentas**. La
prevalencia es del $0{,}1727\ \%$ —una transacción fraudulenta cada 577 legítimas— y los
atributos `V1` a `V28` son obtenidos mediante PCA (principal components analysis), de modo que no hay
interpretación posible de los coeficientes (al aplicar PCA se pierde el significado de las variables).

Los costos de los dos errores son distintos y conocidos: un fraude no detectado cuesta
**500 USD** y una alerta falsa, **5 USD**.

In [ ]:
#@title Carga del conjunto de datos (ejecutar sin modificar) { display-mode: "form" }
from pathlib import Path

URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
CANDIDATAS = [Path("data/creditcard.csv"), Path("../../data/creditcard.csv"),
              Path("creditcard.csv")]

ruta = next((c for c in CANDIDATAS if c.exists()), None)
datos = pd.read_csv(ruta if ruta is not None else URL)
print(f"origen: {ruta if ruta is not None else URL}")

y_todo = datos["Class"].to_numpy()
X_todo = datos.drop(columns=["Class", "Time"]).copy()
X_todo["Amount"] = np.log1p(X_todo["Amount"])     # el monto, en escala logaritmica
X_todo = X_todo.to_numpy()

print(f"n = {len(y_todo):,}   p = {X_todo.shape[1]}   fraudes = {int(y_todo.sum())}")
print(f"prevalencia = {y_todo.mean():.6f} = {100 * y_todo.mean():.4f} %")

### Las funciones de la clase

Las estrategias de corrección ya están implementadas y se entregan tal como se escribieron
durante la exposición. **No corresponde reescribirlas**: el objeto de este laboratorio es
dónde se las llama, no cómo están hechas por dentro.

In [ ]:
def ajustar(X, y, **kw):
    """Regresion logistica, la unica familia que se usa en todo el laboratorio."""
    return LogisticRegression(max_iter=3000, random_state=SEMILLA, **kw).fit(X, y)


def submuestrear(X, y, razon=1.0, semilla=SEMILLA):
    """Descarta negativos al azar, sin reemplazo, hasta la razon pedida (min/may)."""
    r = np.random.default_rng(semilla)
    idx_pos = np.flatnonzero(y == 1)
    idx_neg = np.flatnonzero(y == 0)
    elegidos = r.choice(idx_neg, size=min(int(len(idx_pos) / razon), len(idx_neg)),
                        replace=False)
    idx = r.permutation(np.concatenate([idx_pos, elegidos]))
    return X[idx], y[idx]


def smote(X, y, m=5, razon=1.0, semilla=SEMILLA):
    """SMOTE: interpola entre un minoritario y uno de sus m vecinos MINORITARIOS."""
    r = np.random.default_rng(semilla)
    Xp = X[y == 1]
    g = int(np.sum(y == 0) * razon) - len(Xp)          # cuantas generar
    if g <= 0:
        return X.copy(), y.copy()
    _, ind = NearestNeighbors(n_neighbors=m + 1).fit(Xp).kneighbors(Xp)
    ind = ind[:, 1:]                                   # descartar el propio punto
    base = r.integers(0, len(Xp), size=g)
    vecino = ind[base, r.integers(0, m, size=g)]
    lam = r.random((g, 1))                             # lambda ~ U(0,1)
    sinteticos = Xp[base] + lam * (Xp[vecino] - Xp[base])
    Xn = np.vstack([X, sinteticos])
    yn = np.concatenate([y, np.ones(g, dtype=y.dtype)])
    idx = r.permutation(len(yn))
    return Xn[idx], yn[idx]


def precision_en_k(y, s, k=100):
    """Fraccion de fraudes entre las k transacciones de mayor puntaje."""
    return float(np.mean(y[np.argsort(-s)[:k]]))


def costo(y, s, tau):
    """Costo total en USD de operar con umbral tau."""
    predicho = s >= tau
    fn = int(((y == 1) & ~predicho).sum())
    fp = int(((y == 0) & predicho).sum())
    return fn * COSTO_FN + fp * COSTO_FP


TAU = COSTO_FP / (COSTO_FP + COSTO_FN)
print(f"umbral optimo tau* = {COSTO_FP:.0f} / {COSTO_FP + COSTO_FN:.0f} = {TAU:.6f}")

Y una función que resume una evaluación, para que las dos del ejercicio siguiente sean
**comparables línea por línea**: lo único que va a distinguirlas es el orden en que se
remuestrea y se parte.

In [ ]:
def metricas(y, s):
    """Las cifras con que se compara cada evaluacion.

    Se informan DOS costos, y la distancia entre ellos es lo que separa un modelo
    que ordena mal de uno que ordena bien pero tiene la escala corrida:

    - `costo en tau*`  : el de operar en el umbral que fija la matriz de costos
    - `costo minimo`   : el mejor alcanzable con ese ordenamiento, barriendo tau
    """
    rejilla = np.unique(np.quantile(s, np.linspace(0, 1, 400)))
    costos = np.array([costo(y, s, t) for t in rejilla])
    i = int(np.argmin(costos))
    return {"AP": average_precision_score(y, s),
            "ROC-AUC": roc_auc_score(y, s),
            "precision@100": precision_en_k(y, s, 100),
            "costo en tau*": costo(y, s, TAU),
            "costo minimo": float(costos[i]),
            "tau minimizador": float(rejilla[i]),
            "prevalencia de prueba": float(y.mean())}


def tabla(filas: dict) -> pd.DataFrame:
    """Arma la tabla comparativa a partir de {etiqueta: dict de metricas}."""
    return pd.DataFrame(filas).T

---

# 1. El número que se reporta

El orden que aparece con frecuencia **erróneamente** en código publicado es: cargar, **balancear**, partir,
ajustar, evaluar. Leído en voz alta suena razonable —«primero se arregla el desbalance y
después se trabaja normalmente»— y produce cifras excelentes.

### Ejercicio 1

Implementar ese orden, exactamente como se lo acaba de describir. El escalador se ajusta
sobre el conjunto de ajuste, que es lo único que este procedimiento sí hace bien.

In [ ]:
def evaluar_mal(X, y, semilla=SEMILLA):
    """Remuestrea ANTES de partir. Devuelve las metricas sobre su conjunto de prueba.

    El orden es: SMOTE sobre todo -> particion estratificada 70/30 -> escalado
    ajustado con el conjunto de ajuste -> modelo -> metricas.

    Parámetros
    ----------
    X, y    : el conjunto COMPLETO, sin partir
    semilla : para la particion

    Devuelve
    --------
    dict : la salida de `metricas` sobre el conjunto de prueba de este procedimiento
    """
    # TODO: implementar
    raise NotImplementedError()

In [ ]:
resultado_mal = evaluar_mal(X_todo, y_todo)
for k, v in resultado_mal.items():
    print(f"  {k:24} {v:,.4f}")

---

# 2. El número correcto

La corrección es de dos líneas: **partir primero**, y aplicar SMOTE únicamente al conjunto de
ajuste. El conjunto de prueba queda como llegó, con su prevalencia real y sin un solo punto
sintético.

### Ejercicio 2

Implementar el orden correcto. Todo lo demás —la semilla, la proporción de la partición, el
escalado, el modelo— es idéntico al ejercicio 1: la única diferencia tiene que ser dónde se
llama a `smote`.

In [ ]:
def evaluar_bien(X, y, semilla=SEMILLA):
    """Parte ANTES de remuestrear. Devuelve las metricas sobre prueba intacta.

    El orden es: particion estratificada 70/30 -> escalado ajustado con el conjunto
    de ajuste -> SMOTE SOLO sobre ajuste -> modelo -> metricas sobre prueba.

    Parámetros
    ----------
    X, y    : el conjunto COMPLETO, sin partir
    semilla : para la particion

    Devuelve
    --------
    dict : la salida de `metricas` sobre el conjunto de prueba, que no se tocó
    """
    # TODO: implementar
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN — la brecha entre lo que se reporta y lo que se entrega
resultado_bien = evaluar_bien(X_todo, y_todo)

comparacion = tabla({"remuestreo antes de partir": resultado_mal,
                     "remuestreo despues de partir": resultado_bien})
print(comparacion.to_string(float_format=lambda v: f"{v:,.4f}"))

optimismo = resultado_mal["AP"] - resultado_bien["AP"]
print(f"\noptimismo en average precision: {optimismo:+.4f} "
      f"({100 * optimismo / resultado_bien['AP']:+.1f} % sobre la cifra honesta)")

# La prueba del primer procedimiento no es el problema original: se le cambio la mezcla
assert resultado_mal["prevalencia de prueba"] > 10 * resultado_bien["prevalencia de prueba"]
print("la prevalencia del conjunto de prueba cambio por un factor de "
      f"{resultado_mal['prevalencia de prueba'] / resultado_bien['prevalencia de prueba']:,.0f}")

### Para analizar

1. La average precision del primer procedimiento es mucho mayor. **Hay dos causas
   distintas**, y conviene separarlas: una tiene que ver con **qué observaciones** hay en el
   conjunto de prueba, y la otra con **en qué proporción** están. Enunciar cada una.
2. La línea base de la average precision es la prevalencia del conjunto sobre el que se la
   mide. Con ese dato, **¿cuánto de la mejora aparente se explica sin mirar el modelo?**
3. El costo en dólares del primer procedimiento resulta **mayor**, no menor, pese a que
   todas sus métricas de ordenamiento son mejores. **¿Ese número significa algo?** Indicar
   sobre qué población está calculado, cuántos fraudes contiene esa población y si existe.
   La conclusión vale para las dos direcciones: no es una cifra pesimista, es una cifra sin
   referente.
4. `evaluar_mal` ajusta el escalador sobre el conjunto de ajuste, que es lo correcto. **¿Por
   qué eso no alcanza para salvar el procedimiento?**

**Respuesta:**

---

# 3. La comparación honesta

Una partición única deja la comparación a merced de qué fraudes cayeron de cada lado: son
492 en total, y unos 150 en el conjunto de prueba. La validación cruzada estratificada
promedia sobre $K$ particiones y reduce esa varianza.

La regla del laboratorio se aplica **dentro de cada fold**: se separa el fold de validación,
y recién entonces se escala y se remuestrea con el resto. Un escalador o un SMOTE calculados
antes del bucle contaminarían cada fold de validación con información de sí mismo.

Las cuatro estrategias se declaran con una interfaz común: cada una recibe el conjunto de
ajuste del fold y devuelve **con qué datos ajustar** y **con qué argumentos** construir el
modelo.

In [ ]:
ESTRATEGIAS = {
    "sin intervencion": lambda Xa, ya: (Xa, ya, {}),
    "ponderacion":      lambda Xa, ya: (Xa, ya, {"class_weight": "balanced"}),
    "submuestreo 1:1":  lambda Xa, ya: (*submuestrear(Xa, ya), {}),
    "SMOTE 1:1":        lambda Xa, ya: (*smote(Xa, ya), {}),
}
print("estrategias:", ", ".join(ESTRATEGIAS))

### Ejercicio 3

Implementar la validación cruzada estratificada respetando el orden. En cada fold: separar,
escalar con el escalador ajustado sobre el conjunto de ajuste, aplicar la estrategia
**solo a ese conjunto**, ajustar y evaluar sobre el fold retenido.

`StratifiedKFold` mantiene la proporción de fraudes en cada fold, lo cual con 492 positivos
no es un lujo: con `KFold` común, algún fold podría quedar casi sin fraudes.

In [ ]:
def evaluar_cv(nombre, X, y, K=5):
    """Evalua una estrategia por validacion cruzada estratificada, sin fugas.

    Parámetros
    ----------
    nombre : clave de ESTRATEGIAS
    X, y   : el conjunto COMPLETO, sin partir
    K      : cantidad de folds

    Devuelve
    --------
    dict : promedio sobre los K folds de AP, ROC-AUC, precision@100 y costo USD
    """
    # TODO: implementar
    raise NotImplementedError()

In [ ]:
resultados = {nombre: evaluar_cv(nombre, X_todo, y_todo) for nombre in ESTRATEGIAS}
comparativa = tabla(resultados).drop(columns=["prevalencia de prueba"])
print(comparativa.to_string(float_format=lambda v: f"{v:,.4f}"))

print(f"\nmejor average precision:            {comparativa['AP'].idxmax()}")
print(f"menor costo operando en tau*:       {comparativa['costo en tau*'].idxmin()}")
print(f"menor costo con el umbral ajustado: {comparativa['costo minimo'].idxmin()}")

penalidad = comparativa["costo en tau*"] / comparativa["costo minimo"]
print("\ncuanto se paga por operar en tau* con la escala de cada estrategia:")
for nombre, factor in penalidad.sort_values().items():
    print(f"  {nombre:20} x{factor:6.1f}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10.4, 3.6), constrained_layout=True)
nombres = list(comparativa.index)
posicion = np.arange(len(nombres))
colores = [AZUL, TURQUESA, NARANJA, VIOLETA]

ejes[0].bar(posicion, comparativa["AP"], color=colores)
ejes[0].set_ylabel("Average precision")
ejes[0].set_title("Calidad del ordenamiento")

ancho = 0.38
ejes[1].bar(posicion - ancho / 2, comparativa["costo en tau*"] / 1000, ancho,
            color=NARANJA, label=rf"operando en $\tau^\star={TAU:.4f}$")
ejes[1].bar(posicion + ancho / 2, comparativa["costo minimo"] / 1000, ancho,
            color=OLIVA, label="con el umbral ajustado")
ejes[1].set_ylabel("Costo en el fold, miles de USD")
ejes[1].set_title("Costo: la escala contra el ordenamiento")
ejes[1].legend(fontsize=8)

for eje in ejes:
    eje.set_xticks(posicion)
    eje.set_xticklabels(nombres, rotation=20, ha="right", fontsize=8)
plt.show()

### HACER

1. Comparar la average precision de `SMOTE 1:1` acá con la del ejercicio 1. **¿Cuál de las
   dos describe lo que el modelo va a hacer con transacciones nuevas?**


**Respuesta:**

**Respuesta:**

---

## Cierre

- **El remuestreo pertenece al interior de la partición.** Aplicado antes, contamina el
  conjunto de evaluación con puntos derivados de los de ajuste y le cambia la prevalencia:
  las dos cosas inflan la cifra que se reporta.
- **Una métrica sin su población no significa nada.** La average precision se lee contra la
  prevalencia del conjunto sobre el que se la midió, y un procedimiento que modifica esa
  prevalencia hace incomparables las cifras de antes y después.
- **Balancear no es gratis ni obligatorio.** Bajo un protocolo honesto, la estrategia sin
  intervención no queda última, y la que gana en ordenamiento no es necesariamente la que
  minimiza el costo.
- **Una escala rota se repara y el ordenamiento no.** Las tres reparaciones son monótonas
  crecientes: devuelven el costo a su mínimo sin mover el ROC-AUC ni la average precision.
- **El calibrador es un modelo más**, y se estima con datos que el modelo no vio.

## Para seguir explorando

Sin obligación y fuera del tiempo de clase:

1. **La razón de remuestreo.** `submuestrear` y `smote` aceptan `razon`. Repetir la
   comparación de la sección 3 con $0{,}05$, $0{,}2$ y $1{,}0$, y verificar si balancear del
   todo es lo mejor.
2. **El umbral como estrategia.** Agregar a `ESTRATEGIAS` una entrada que no toque los datos
   y en su lugar elija $\tau$ por validación cruzada minimizando el costo. Comparar contra
   las cuatro anteriores.
3. **La partición temporal.** El conjunto trae `Time`, que acá se descarta. Rehacer la
   sección 3 partiendo por tiempo en lugar de al azar, y explicar por qué el resultado cambia
   y cuál de los dos protocolos se parece más a la operación real.